# 📊 Model Download Monitor

Track the progress of your AI model downloads in real-time.

In [ ]:
import subprocess
import time
from datetime import datetime
from IPython.display import display, HTML, clear_output

def monitor_ollama_models():
    """Monitor Ollama model download progress"""
    target_models = [
        "qwen2.5-coder:7b", "qwen2.5-coder:14b", "codellama:7b",
        "deepseek-coder:v2", "deepseek-coder-v3:35b-q4_K_M",
        "aixcoder-7b", "phi4", "starcoder2:15b", "gemma2:9b", "phi3.5"
    ]
    
    try:
        result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
        if result.returncode == 0:
            output_lines = result.stdout.strip().split('\n')
            installed_models = []
            
            if len(output_lines) > 1:  # Skip header
                for line in output_lines[1:]:
                    if line.strip():
                        model_name = line.split()[0]
                        installed_models.append(model_name)
            
            html = f"""
            <div style="background: linear-gradient(135deg, #2196f3 0%, #21cbf3 100%); 
                        color: white; padding: 20px; border-radius: 10px; font-family: 'Segoe UI', Arial, sans-serif;">
                <h2 style="margin-top: 0; text-align: center;">🤖 AI Model Download Status</h2>
                <p style="text-align: center; margin-bottom: 20px;"><strong>Last Updated:</strong> {datetime.now().strftime('%H:%M:%S')}</p>
                
                <div style="background: rgba(255,255,255,0.1); padding: 15px; border-radius: 8px;">
                    <h3 style="margin-top: 0; color: #ffeb3b;">📋 Model Status ({len(installed_models)}/{len(target_models)} Ready)</h3>
                    <ul style="list-style: none; padding: 0;">
            """
            
            for model in target_models:
                status = "✅" if model in installed_models else "⏳"
                size_info = ""
                if model in installed_models:
                    # Find size info from ollama list output
                    for line in output_lines[1:]:
                        if model in line:
                            parts = line.split()
                            if len(parts) >= 3:
                                size_info = f" ({parts[2]})"
                            break
                
                html += f"<li style='margin: 5px 0;'><strong>{status} {model}</strong>{size_info}</li>"
            
            html += """
                    </ul>
                </div>
            </div>
            """
            
            return html, len(installed_models), len(target_models)
        else:
            return "<p>❌ Could not connect to Ollama</p>", 0, len(target_models)
    except Exception as e:
        return f"<p>❌ Error: {e}</p>", 0, len(target_models)

# Initial check
html, ready_count, total_count = monitor_ollama_models()
display(HTML(html))
print(f"\n🎯 Progress: {ready_count}/{total_count} models ready")
print("\n💡 Run this cell periodically to check download progress!")

## Auto-Refresh Monitor

Run this cell for continuous monitoring (stops after all models are downloaded):

In [ ]:
import time

# Auto-refresh monitoring loop
max_checks = 30  # Maximum number of checks
check_interval = 30  # Seconds between checks

for i in range(max_checks):
    clear_output(wait=True)
    html, ready_count, total_count = monitor_ollama_models()
    display(HTML(html))
    
    print(f"\n📊 Check {i+1}/{max_checks} - Progress: {ready_count}/{total_count} models ready")
    
    if ready_count == total_count:
        print("\n🎉 All models downloaded! Setup complete!")
        break
    
    if i < max_checks - 1:  # Don't wait after the last check
        print(f"\n⏳ Waiting {check_interval} seconds for next check...")
        time.sleep(check_interval)

print("\n✅ Monitoring complete!")